<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day25_sft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.2 MB/s eta 0:00:00


In [ ]:
!pip install -q trl peft transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00


In [ ]:
!pip install -q -U torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 13.3 MB/s eta 0:00:00


In [ ]:
import trl, peft, transformers, torch
print("trl", trl.__version__)
print("peft", peft.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__, torch.cuda.is_available())

trl 1.10.0
peft 0.20.0
transformers 5.13.1
torch 2.11.0+cu128 True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/Colab\ Notebooks

In [ ]:
BASE = "/content/drive/MyDrive/Colab Notebooks/"
!ls -lh "{BASE}"



In [ ]:
import json
with open(BASE + "pairs.json", encoding="utf-8") as f:
    raw = json.load(f)
print(f"{len(raw)} 筆")


In [ ]:
# ===== Cell 4: 轉成訓練格式 =====
from datasets import Dataset

SYSTEM = "あなたは会議の議事録から構造化データを抽出するアシスタントです。指定されたJSON形式のみを出力してください。"

formatted = [{
    "prompt": [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": p["note"]},
    ],
    "completion": [
        {"role": "assistant",
         "content": json.dumps(p["json"], ensure_ascii=False)},
    ],
} for p in raw.values()]

dataset = Dataset.from_list(formatted)

# 方向檢查
print(dataset[0]["prompt"][1]["content"][:20])       # 日文
print(dataset[0]["completion"][0]["content"][:30])   # {"meeting_title"...

In [ ]:

split = dataset.train_test_split(test_size=20, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(len(train_ds), len(eval_ds))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(model.device)

In [ ]:
print(model.device)

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["prompt"] + dataset[0]["completion"],
    tokenize=False
)
print(text[:200])
print("...")
print(repr(text[-80:]))

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="/content/sft_out",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",
    bf16=False,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()

In [ ]:
trainer.save_model(BASE + "lora_adapter_day25")